# IG 品牌帳號探索與資訊爬取

從網紅 CSV 的 `text` 擷取 @username，排除資料庫網紅，保留來源證據並蒐集 IG profile。

`brand_id` 為小寫 IG username（不含 @），數字 ID 另存 `instagram_user_id`；`brand_name` 為顯示名稱。三個數量欄位為粉絲數、追蹤數、全部已發佈貼文數。`business_type_of_the_brand` 使用 IG business category，缺值留空，不猜測。

被提及帳號可能是人物或活動，必須人工確認品牌；business account 本身不足以證明品牌。

In [16]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "scripts" / "ig_brand_crawler_scripts" / "ig_brand_pipeline.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("請從專案根目錄或其子目錄開啟此 notebook。")
SCRIPT_DIR = ROOT / "scripts" / "ig_brand_crawler_scripts"
sys.path.insert(0, str(SCRIPT_DIR))
from ig_brand_pipeline import (
    discover_candidates, read_csv, make_loader, crawl_profiles,
    export_profiles, load_attempts,
)
SOURCE_DIR = Path("/home/ftp_246/data_5/多模態2026_ig_top200/Metadata_by_Influencer")
OUTPUT_DIR = ROOT / "data" / "ig_brand"
print("輸出資料夾：", OUTPUT_DIR)

輸出資料夾： /home/yingtzudai/Multimodal_SPoD_Video/data/ig_brand


## 1. 擷取候選帳號（不連線 IG）

用來源檔名及全部 `post_owner.username` 建立排除名單。支援半形／全形 @，帳號轉小寫，排除一般 email 與不合法帳號；句尾句點去除。重跑保留 `brand_review.csv` 既有判斷。

`mention_count` 為出現次數，`post_count` 依貼文 ID 去重，`influencer_count` 為提及的不同網紅數。排序先依網紅數，再依提及次數。

In [17]:
summary = discover_candidates(SOURCE_DIR, OUTPUT_DIR)
summary

{'files': 199,
 'posts': 19443,
 'influencers': 199,
 'candidates': 2564,
 'excluded_mentions': 546}

In [18]:
candidates = pd.read_csv(OUTPUT_DIR / "brand_candidates.csv", dtype={"brand_id": str})
display(candidates.head(30))

,brand_id,mention_count,post_count,influencer_count,profile_url
0,shopee_tw,36,36,29,https://www.instagram.com/shopee_tw/
1,7eleventw,26,26,17,https://www.instagram.com/7eleventw/
2,watsonstaiwan,16,16,14,https://www.instagram.com/watsonstaiwan/
3,legotw_official,13,13,13,https://www.instagram.com/legotw_official/
4,dysonbeauty,12,12,11,https://www.instagram.com/dysonbeauty/
5,ubereats_tw,12,11,10,https://www.instagram.com/ubereats_tw/
6,netflixtw,22,22,9,https://www.instagram.com/netflixtw/
7,insta360,20,20,8,https://www.instagram.com/insta360/
8,samsungtaiwan,18,18,8,https://www.instagram.com/samsungtaiwan/
9,nivea_tw,12,12,8,https://www.instagram.com/nivea_tw/


In [19]:
evidence = pd.read_csv(OUTPUT_DIR / "mention_evidence.csv", dtype=str, keep_default_na=False)
if not candidates.empty:
    inspect_id = candidates.iloc[0]["brand_id"]  # 可換成想查看的帳號
    display(evidence.loc[evidence.brand_id == inspect_id].head(10))

,brand_id,source_file,source_row,influencer_username,post_id,short_code,occurrences,text
71,shopee_tw,2uncle987-FullVideoInfo.csv,156,2uncle987,791820440614520,DON9xgYD53C,1,不用再懷疑我拍一堆短影音\n只有流量但卻不能變現🤣\n那是因為你沒有使用「蝦皮分潤計畫」😎\...
177,shopee_tw,602avenue-FullVideoInfo.csv,30,602avenue,959506466750463,DJrGAbhTSxu,1,韓國人VS台式馬卡龍\n為了尋找道地的台式馬卡龍....\n去了很多地方，但朋友說要去北車找...
203,shopee_tw,602avenue-FullVideoInfo.csv,135,602avenue,1648914449891960,DQl_m6AEVjS,1,韓國卡片真的要自己插進去喔....\n還有什麼台韓文化差異呢？！\n大家可以底下留言分享看看...
227,shopee_tw,87acup-FullVideoInfo.csv,25,87acup,1984151562516205,DIbExDjPmJ4,1,每個學生的回憶裡一定都有這樣的補習班老師！\n以前去北車大補習班吸著濃密的二氧化碳\n跟隔壁...
234,shopee_tw,87acup-FullVideoInfo.csv,64,87acup,911772547910453,DNiBkTlPX2C,1,唯有保持善念，才能避免厄運降臨...\n從踏進這個變裝派對的那刻起\n鬼ㄒㄆ就盯上你了！\n...
1306,shopee_tw,baxuan_ig-FullVideoInfo.csv,52,baxuan_ig,878528908494263,DSHYrpkk7_P,1,阿嬤你好棒喔\n\n蝦皮購物推出全新功能蝦皮短影音啦！\n不管是買家賣家、年輕人還是阿公阿嬤...
1636,shopee_tw,children0704-FullVideoInfo.csv,17,children0704,26489578847316340,DMzqqGfpI0U,1,我保養最討厭一堆瓶瓶罐罐，又搞不清楚步驟！\n【妮維雅男士水活保濕系列】就是簡單！就是爽！\...
1773,shopee_tw,da_chien_huang-FullVideoInfo.csv,26,da_chien_huang,1233798315050489,DSR6x5QEbDz,1,真的不想裝了\n\n人生已經夠複雜\n想要回歸簡單？\n讓蝦皮陪你一起不裝了！\n\n蝦皮最...
1849,shopee_tw,dannybeeech-FullVideoInfo.csv,58,dannybeeech,3227661334082180,DQwZQBaDw-7,1,影片看到最後！有香的！\n3CE全新的柔光小粉管我光看官方照片就被燒，而且國外網站評價也超好...
2124,shopee_tw,energydessert2019-FullVideoInfo.csv,97,energydessert2019,1488598062984623,DNhU7zvJuPC,1,中元拜拜每年都毫無新意，拜完了供品乏人問津？\n今年不一樣！蝦皮中元節期間限定好拜必buy特...


## 2. 安裝與登入設定

建議選擇專案 `.venv` kernel。需要時取消下方 `%pip` 指令的註解安裝套件。

部分欄位可能需登入；可載入自己已建立且可信的 Instaloader session。若沒有 session，在專案根目錄的終端機執行（替換 YOUR_USERNAME）：

```bash
.venv/bin/instaloader --login YOUR_USERNAME --sessionfile /home/yingtzudai/Multimodal_SPoD_Video/data/ig_brand/session-YOUR_USERNAME
```

不指定下載目標，只互動式登入。密碼及雙重驗證於終端輸入，不放在 notebook。Session 包含登入憑證，請勿分享或提交。

參考：[Profile 欄位](https://instaloader.github.io/module/structures.html)、[載入 session](https://instaloader.github.io/module/instaloader.html#instaloader.Instaloader.load_session_from_file)。

In [20]:
# %pip install -r {SCRIPT_DIR / "requirements-ig-brand.txt"}

LOGIN_USERNAME = "cdna_ying"
SESSION_FILE= OUTPUT_DIR / "session-cdna_ying"

## 3. 小批次爬取與續跑

預設 `RUN_CRAWL=False`，避免執行整本時自動大量發出請求。準備好後設為 True，先試 5 個帳號；間隔 15–30 秒，Instaloader 可能額外等待。

重跑跳過已嘗試帳號，`RETRY_FAILED=True` 可重試失敗或 partial。`ONLY_CONFIRMED_BRANDS=True` 只爬人工確認品牌。每個帳號完成即寫入 checkpoint；遇限流、登入或連線錯誤停止當批。

先前未登入測試 `shopee_tw` 收到 429 Too Many Requests，尚未取得 profile 資訊。請等待限流解除再試；登入也不保證解除限流。若要更新成功帳號的數字，請另設新輸出資料夾並建立候選與審核清單。

In [21]:
SESSION_FILE = OUTPUT_DIR / "session-cdna_ying"
print(SESSION_FILE.is_file())

True


In [22]:
RUN_CRAWL = True
MAX_PROFILES = 1
ONLY_CONFIRMED_BRANDS = False
RETRY_FAILED = True

if RUN_CRAWL:
    decisions = {r["brand_id"]: r["decision"] for r in read_csv(OUTPUT_DIR / "brand_review.csv")}
    usernames = [r["brand_id"] for r in read_csv(OUTPUT_DIR / "brand_candidates.csv")
                 if decisions.get(r["brand_id"]) != "not_brand"
                 and (not ONLY_CONFIRMED_BRANDS or decisions.get(r["brand_id"]) == "brand")]
    loader = make_loader(LOGIN_USERNAME, SESSION_FILE)
    attempts = crawl_profiles(
        usernames, loader, SOURCE_DIR, OUTPUT_DIR,
        max_profiles=MAX_PROFILES, min_delay=15, max_delay=30,
        retry_failed=RETRY_FAILED,
    )
else:
    print("尚未連線 IG。設定 RUN_CRAWL=True 可執行小批次爬取。")

Loaded session from /home/yingtzudai/Multimodal_SPoD_Video/data/ig_brand/session-cdna_ying.
shopee_tw: error
本批次停止，請檢查 profile_attempts.jsonl 的錯誤後再續跑。


### 查看錯誤與缺值

JSONL 保留全部歷史；以下顯示每個帳號最新一次狀態。`partial` 表示至少一個目標欄位缺值，可在 `missing_fields` 查看原因。

In [23]:
all_attempts = load_attempts(OUTPUT_DIR / "profile_attempts.jsonl")
latest_attempts = {r["brand_id"]: r for r in all_attempts}
if latest_attempts:
    display(pd.DataFrame(latest_attempts.values()).reindex(
        columns=["brand_id", "status", "fetched_at_utc", "error_type", "error", "missing_fields"]))
else:
    print("尚無爬取紀錄。")

,brand_id,status,fetched_at_utc,error_type,error,missing_fields
0,shopee_tw,error,2026-11-13T03:06:13.219620+00:00,AbortDownloadException,"400 Bad Request - ""fail"" status, message ""feed...",NaN


## 4. 人工確認品牌與匯出

編輯 `data/ig_brand/brand_review.csv` 的 `decision`：`pending` 尚未確認、`brand` 確認品牌、`not_brand` 排除、`unsure` 待查。`notes` 可記判斷依據；不要更改或重複 `brand_id`。

依來源貼文、簡介、官網、帳號頁面等判斷，儲存 CSV 後執行下一格。`brands.csv` 僅匯出人工確認為品牌且取得資料的帳號，正好六欄；缺值留空、0 保留。沒有符合資料時只產生標題列。更新失敗不會抹除舊有可用資料。

In [24]:
profiles = export_profiles(SOURCE_DIR, OUTPUT_DIR)
brands = pd.read_csv(OUTPUT_DIR / "brands.csv", dtype={"brand_id": str}, keep_default_na=False)
print(f"已取得 profile：{len(profiles)}；已確認且有資料的品牌：{len(brands)}")
display(brands.head(30))

已取得 profile：0；已確認且有資料的品牌：0


,brand_id,brand_name,number_of_followers,number_of_followees,number_of_published_posts,business_type_of_the_brand


全部產物位於 `data/ig_brand/`：

- `influencer_usernames.csv`：排除網紅名單。
- `brand_candidates.csv`：候選帳號、提及統計與 IG 網址。
- `mention_evidence.csv`：來源檔、貼文 ID、原文與 CSV 邏輯列號。
- `brand_review.csv`：人工品牌判斷。
- `profile_attempts.jsonl`：請求結果、UTC 時間、錯誤及缺值原因。
- `candidate_profiles.csv`：最新可用候選資料，含 IG 數字 ID、簡介等。
- `brands.csv`：確認品牌的六欄資料。

實作位於同資料夾 `ig_brand_pipeline.py`，本地假資料測試位於 `test_ig_brand_pipeline.py`。來源資料庫全程只讀。